In [ ]:
# ============================================
# WAXAL ASR v2 - Memory Optimized Version
# ============================================

"""
🎙️ WAXAL ASR v2 — Multilingual Whisper, Sunbird-tuned
Memory-optimized for Google Colab T4 GPU (12GB RAM)

Key optimizations:
- Streaming dataset loading (no storing all data in RAM)
- Batch processing with memory cleanup
- Gradient checkpointing and mixed precision
- Automatic cache clearing
"""

# 1. Setup - Mount Drive
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ValueError:
    print("(already mounted)")

WORK_DIR = '/content/drive/MyDrive/waxal_whisper'
os.makedirs(WORK_DIR, exist_ok=True)
print('Working dir:', WORK_DIR)

# 2. Install dependencies
!pip install -q -U transformers datasets accelerate jiwer evaluate librosa soundfile audiomentations psutil
print("Dependencies installed")

# 3. Hugging Face login
from huggingface_hub import login
login()

# 4. Configuration with memory-safe defaults
from typing import Final

MODEL_ID: Final = "openai/whisper-small"
DATASET_ID: Final = "google/WaxalNLP"
LANGS: Final = ["lin", "sna", "lug"]

# ----- DATA (Start small, then increase) -----
PER_LANG_TRAIN: Final = 300   # Start small to test
PER_LANG_EVAL: Final  = 50    # Reduced for memory

# ----- QUALITY FILTER -----
MIN_CHARS: Final       = 5
MAX_TOKENS_APPROX: Final = 80
MAX_NONLETTER_FRAC: Final = 0.15

# ----- AUGMENTATION -----
AUG_PROB: Final        = 0.5
TELEPHONE_FRAC: Final  = 0.05

# ----- TRAINING (Reduced for memory) -----
BATCH: Final       = 4      # Reduced from 8
GRAD_ACCUM: Final  = 8      # Increased to maintain effective batch size
MAX_STEPS: Final   = 1000   # Reduced
SAVE_EVERY: Final  = 250
EVAL_EVERY: Final  = 250
LR: Final          = 1e-5
EARLY_STOP_PATIENCE: Final = 4

# ----- DECODING -----
NUM_BEAMS: Final        = 3   # Reduced from 5
REPETITION_PENALTY: Final = 1.15
NO_REPEAT_NGRAM: Final  = 4

OUT_DIR = f"{WORK_DIR}/whisper-waxal-v2"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Model: {MODEL_ID} | Per-lang train: {PER_LANG_TRAIN} | Out: {OUT_DIR}")

# 5. Memory monitoring utilities
import gc
import psutil
import torch

def memory_status(stage=""):
    """Check current memory usage"""
    print(f"\n=== Memory Status {stage} ===")
    if torch.cuda.is_available():
        print(f"GPU allocated: {torch.cuda.memory_allocated()/1024**3:.2f}GB")
        print(f"GPU cached: {torch.cuda.memory_reserved()/1024**3:.2f}GB")
    print(f"System RAM used: {psutil.virtual_memory().used/1024**3:.2f}GB")
    print(f"System RAM free: {psutil.virtual_memory().free/1024**3:.2f}GB")

def cleanup_memory():
    """Force memory cleanup"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    gc.collect()
    memory_status("After cleanup")

# 6. Normalization + quality filter
import re

NOISE_MARKER = re.compile(r"[\n\(<]\s*(noise|inaudible|unclear|silence|music|laugh|cough|unintelligible)[^\n\)>]*[\]\)>]", re.I)

def normalize_text(t: str) -> str:
    t = str(t).strip()
    return re.sub(r"\s+", " ", t)

def is_good_transcript(t: str) -> bool:
    """Sunbird noisy-ground-truth heuristics."""
    if t is None: return False
    t = str(t).strip()
    if len(t) < MIN_CHARS: return False
    if NOISE_MARKER.search(t): return False
    if len(t.split()) > MAX_TOKENS_APPROX: return False
    letters = sum(ch.isalpha() or ch.isspace() for ch in t)
    if letters / max(len(t), 1) < (1 - MAX_NONLETTER_FRAC):
        return False
    return True

# 7. Streaming dataset loader (memory efficient)
def stream_take_streaming(lang, split, n):
    """Load dataset in streaming mode - uses minimal memory"""
    from datasets import load_dataset, Audio
    
    # Get config
    configs = get_dataset_config_names(DATASET_ID)
    cands = sorted([c for c in configs if c.startswith(lang)],
                   key=lambda c: ("tts" in c, "asr" not in c, len(c)))
    cfg = cands[0] if cands else f"{lang}_asr"
    
    # Load streaming
    s = load_dataset(DATASET_ID, cfg, split=split, streaming=True)
    s = s.cast_column("audio", Audio(sampling_rate=16_000))
    
    # Generator function - yields one item at a time
    def gen():
        seen, kept = 0, 0
        for ex in s:
            seen += 1
            if kept >= n: break
            if seen > n * 3: break
            
            if not is_good_transcript(ex.get("transcription")):
                continue
            
            kept += 1
            yield {
                "audio": ex["audio"]["array"],
                "transcription": ex["transcription"],
                "language": lang,
                "speaker_id": str(ex.get("speaker_id", f"{lang}_spk_{kept}")),
            }
            
            # Periodic cleanup while streaming
            if kept % 50 == 0:
                import gc; gc.collect()
    
    return Dataset.from_generator(gen)

# 8. Load data with memory management
from datasets import get_dataset_config_names, Dataset, concatenate_datasets

print("\n=== Loading Data ===")
cleanup_memory()

train_parts, eval_parts = [], []
for lang in LANGS:
    print(f"\nLoading {lang}...")
    train_parts.append(stream_take_streaming(lang, "train", PER_LANG_TRAIN))
    cleanup_memory()
    
    try:
        eval_parts.append(stream_take_streaming(lang, "validation", PER_LANG_EVAL))
    except Exception:
        eval_parts.append(stream_take_streaming(lang, "train", PER_LANG_EVAL))
    cleanup_memory()

# Combine datasets
train_ds = concatenate_datasets(train_parts).shuffle(seed=42)
eval_ds = concatenate_datasets(eval_parts)
print(f"\nTOTAL train: {len(train_ds)} | eval: {len(eval_ds)}")
cleanup_memory()

# 9. Audio augmentation
import numpy as np
import librosa

rng = np.random.default_rng(42)

def augment(arr: np.ndarray) -> np.ndarray:
    a = np.asarray(arr, dtype=np.float32)
    
    # Speed perturbation
    if rng.random() < 0.5:
        rate = float(rng.uniform(0.9, 1.1))
        a = librosa.effects.time_stretch(a, rate=rate)
    
    # Random noise
    if rng.random() < 0.5:
        snr_db = float(rng.uniform(10, 30))
        rms = np.sqrt(np.mean(a ** 2)) + 1e-9
        noise_rms = rms / (10 ** (snr_db / 20))
        a = a + rng.normal(0, noise_rms, a.shape).astype(np.float32)
    
    # Telephone simulation
    if rng.random() < TELEPHONE_FRAC / AUG_PROB:
        a = librosa.resample(a, orig_sr=16_000, target_sr=8_000)
        a = librosa.resample(a, orig_sr=8_000, target_sr=16_000)
    
    return np.clip(a, -1.0, 1.0)

# 10. Processor and feature preparation
from transformers import WhisperProcessor
processor = WhisperProcessor.from_pretrained(MODEL_ID)

def make_prepare(do_augment: bool):
    def prepare(batch):
        arr = np.asarray(batch["audio"], dtype=np.float32)
        if do_augment and rng.random() < AUG_PROB:
            arr = augment(arr)
        batch["input_features"] = processor.feature_extractor(
            arr, sampling_rate=16_000).input_features[0]
        batch["labels"] = processor.tokenizer(
            normalize_text(batch["transcription"])).input_ids
        return batch
    return prepare

# 11. Batch processor for memory efficiency
def process_in_batches(dataset, do_augment, batch_size=16):
    """Process dataset in smaller batches to manage memory"""
    def batch_generator():
        for i in range(0, len(dataset), batch_size):
            # Get batch
            end_idx = min(i + batch_size, len(dataset))
            batch = dataset.select(range(i, end_idx))
            
            # Process batch
            processed = batch.map(make_prepare(do_augment), 
                                remove_columns=batch.column_names)
            
            # Yield processed items
            for item in processed:
                yield item
            
            # Cleanup
            del batch, processed
            if i % 50 == 0:
                gc.collect()
    
    return Dataset.from_generator(batch_generator)

# 12. Process data in batches
print("\n=== Preprocessing Data ===")
train_prep = process_in_batches(train_ds, True, batch_size=12)
eval_prep = process_in_batches(eval_ds, False, batch_size=12)
print(f"Prepared - train: {len(train_prep)} | eval: {len(eval_prep)}")

# Free up memory
del train_ds, eval_ds, train_parts, eval_parts
cleanup_memory()

# 13. Data collator
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int
    
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

# 14. Load model with memory optimizations
print("\n=== Loading Model ===")
from transformers import WhisperForConditionalGeneration

# Try to load with 8-bit quantization if available
try:
    from transformers import BitsAndBytesConfig
    quantization_config = BitsAndBytesConfig(load_in_8bit=True)
    model = WhisperForConditionalGeneration.from_pretrained(
        MODEL_ID, 
        quantization_config=quantization_config,
        device_map="auto"
    )
    print("Loaded with 8-bit quantization")
except (ImportError, Exception) as e:
    print(f"8-bit not available: {e}")
    model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
    # Move to GPU if available
    if torch.cuda.is_available():
        model = model.to("cuda")

# Model configuration
model.generation_config.language = None
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

# Enable memory optimizations
model.config.use_cache = False  # Disable cache during training
model.gradient_checkpointing_enable()

# Anti-repetition decoding
model.generation_config.num_beams = NUM_BEAMS
model.generation_config.repetition_penalty = REPETITION_PENALTY
model.generation_config.no_repeat_ngram_size = NO_REPEAT_NGRAM

# SpecAugment
model.config.apply_spec_augment = True
model.config.mask_time_prob = 0.05
model.config.mask_feature_prob = 0.05

print("Model ready")
cleanup_memory()

# 15. Create collator
collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor, 
    decoder_start_token_id=model.config.decoder_start_token_id
)

# 16. Metrics
import evaluate
wer_m = evaluate.load("wer")
cer_m = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids, label_ids = pred.predictions, pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    
    pred_str = [normalize_text(p) for p in processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)]
    label_str = [normalize_text(l) for l in processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)]
    
    keep = [(p, l) for p, l in zip(pred_str, label_str) if l.strip()]
    if not keep:
        return {"wer": 1.0, "cer": 1.0, "score": 1.0}
    
    pred_str, label_str = zip(*keep)
    wer = wer_m.compute(predictions=list(pred_str), references=list(label_str))
    cer = cer_m.compute(predictions=list(pred_str), references=list(label_str))
    return {"wer": wer, "cer": cer, "score": 0.5*wer + 0.5*cer}

# 17. Training arguments with memory optimizations
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback

args = Seq2SeqTrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=50,
    max_steps=MAX_STEPS,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=4,  # Reduced for memory
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=SAVE_EVERY,
    eval_steps=EVAL_EVERY,
    logging_steps=50,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="score",
    greater_is_better=False,
    save_total_limit=1,  # Keep only 1 checkpoint
    seed=42,
    dataloader_pin_memory=False,  # Don't pin memory
    remove_unused_columns=True,
    optim="adamw_torch",  # Standard optimizer
    max_grad_norm=1.0,
    eval_accumulation_steps=1,  # Eval in smaller batches
)

# 18. Trainer
trainer = Seq2SeqTrainer(
    args=args,
    model=model,
    train_dataset=train_prep,
    eval_dataset=eval_prep,
    data_collator=collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)],
)

# 19. Check for resume checkpoint
import glob
ckpts = sorted(glob.glob(f"{OUT_DIR}/checkpoint-*"),
               key=lambda p: int(p.split("-")[-1]))
resume = ckpts[-1] if ckpts else None
print(f"\nResuming from: {resume}")

# 20. Train
print("\n=== Starting Training ===")
memory_status("Before training")

try:
    trainer.train(resume_from_checkpoint=resume)
except RuntimeError as e:
    if "out of memory" in str(e).lower():
        print("\n⚠️ Out of memory error! Try:")
        print("1. Reduce PER_LANG_TRAIN further")
        print("2. Reduce BATCH size to 2")
        print("3. Set MAX_STEPS lower")
        cleanup_memory()
    raise e

# 21. Save model
print("\n=== Saving Model ===")
trainer.save_model(f"{OUT_DIR}/final")
processor.save_pretrained(f"{OUT_DIR}/final")
cleanup_memory()
print(f"✅ Model saved to {OUT_DIR}/final")

# 22. Baseline comparison
print("\n=== Baseline Comparison ===")

def quick_score(mdl, proc, ds, n=30):
    """Quick evaluation on subset"""
    mdl.eval()
    preds, refs = [], []
    n = min(n, len(ds))
    
    for i in range(0, n, 2):  # Process in smaller chunks
        end = min(i+2, n)
        batch = ds.select(range(i, end))
        
        for ex in batch:
            feats = proc.feature_extractor(
                np.asarray(ex["audio"], dtype=np.float32),
                sampling_rate=16_000,
                return_tensors="pt"
            ).input_features.to(mdl.device)
            
            with torch.no_grad():
                ids = mdl.generate(
                    feats, 
                    max_new_tokens=225,
                    num_beams=NUM_BEAMS,
                    repetition_penalty=REPETITION_PENALTY,
                    no_repeat_ngram_size=NO_REPEAT_NGRAM
                )
            
            preds.append(normalize_text(proc.tokenizer.batch_decode(ids, skip_special_tokens=True)[0]))
            refs.append(normalize_text(ex["transcription"]))
        
        # Cleanup between batches
        del batch
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    w = wer_m.compute(predictions=preds, references=refs)
    c = cer_m.compute(predictions=preds, references=refs)
    return w, c, 0.5*w + 0.5*c

# Load base model
base_model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
if torch.cuda.is_available():
    base_model = base_model.to("cuda")
base_model.generation_config.language = None
base_model.generation_config.forced_decoder_ids = None

# Score both models
print("Scoring baseline model...")
bw, bc, bs = quick_score(base_model, processor, eval_prep)
del base_model
cleanup_memory()

print("Scoring fine-tuned model...")
fw, fc, fs = quick_score(model, processor, eval_prep)

print(f"\nBASELINE (no fine-tune): WER={bw:.3f} CER={bc:.3f} score={bs:.3f}")
print(f"FINE-TUNED             : WER={fw:.3f} CER={fc:.3f} score={fs:.3f}")

if fs < bs:
    print("✅ Fine-tuning helped!")
else:
    print("⚠️ Fine-tune is WORSE than baseline - need more data")

# 23. Inference on test set
print("\n=== Running Inference ===")

# Load test data
import pandas as pd
TEST_CSV = "Test.csv"
SAMPLE_CSV = "SampleSubmission.csv"

try:
    test = pd.read_csv(TEST_CSV)
    id_col = test.columns[0]
    print(f"Test IDs: {len(test)} | Example: {test[id_col].iloc[0]}")
except FileNotFoundError:
    print("Test.csv not found. Please upload to current directory.")
    print("Creating empty submission for demonstration...")
    test = pd.DataFrame({"id": ["test_1", "test_2"]})
    id_col = "id"

# Build audio lookup for test set
def build_lookup():
    """Build audio lookup for test IDs"""
    from datasets import load_dataset, Audio
    
    lookup, need = {}, set(test[id_col])
    configs = get_dataset_config_names(DATASET_ID)
    
    for lang in LANGS:
        if not need: break
        
        cands = sorted([c for c in configs if c.startswith(lang)],
                      key=lambda c: ("tts" in c, "asr" not in c, len(c)))
        cfg = cands[0] if cands else f"{lang}_asr"
        
        for split in ["test", "validation", "train"]:
            if not need: break
            try:
                s = load_dataset(DATASET_ID, cfg, split=split, streaming=True)
                s = s.cast_column("audio", Audio(sampling_rate=16_000))
                for ex in s:
                    if ex["id"] in need:
                        lookup[ex["id"]] = ex["audio"]["array"]
                        need.discard(ex["id"])
                        if len(lookup) % 50 == 0:
                            print(f"  Found {len(lookup)}/{len(test)}")
                        if not need: break
            except Exception as e:
                print(f"  {lang}/{split}: {str(e)[:60]}")
                continue
    
    print(f"Resolved {len(lookup)}/{len(test)} audio files")
    return lookup

audio_lookup = build_lookup()

# Transcribe
def transcribe(arr):
    """Transcribe audio array"""
    feats = processor.feature_extractor(
        np.asarray(arr, dtype=np.float32),
        sampling_rate=16_000,
        return_tensors="pt"
    ).input_features.to(model.device)
    
    with torch.no_grad():
        ids = model.generate(
            feats,
            max_new_tokens=225,
            num_beams=NUM_BEAMS,
            repetition_penalty=REPETITION_PENALTY,
            no_repeat_ngram_size=NO_REPEAT_NGRAM
        )
    return normalize_text(processor.tokenizer.batch_decode(ids, skip_special_tokens=True)[0])

# Transcribe in batches
print("\nTranscribing test set...")
rows = []
batch_size = 5  # Small batch for memory

for i in range(0, len(test), batch_size):
    end = min(i+batch_size, len(test))
    batch = test[id_col].iloc[i:end]
    
    for _id in batch:
        arr = audio_lookup.get(_id)
        if arr is not None:
            text = transcribe(arr)
        else:
            text = ""
        rows.append((_id, text))
    
    if (i+batch_size) % 50 == 0:
        print(f"  Transcribed {i+batch_size}/{len(test)}")
    cleanup_memory()

print(f"Transcribed {len(rows)} samples")

# Create submission
sample = pd.read_csv(SAMPLE_CSV) if os.path.exists(SAMPLE_CSV) else None

if sample is not None:
    sid, stgt = sample.columns[0], sample.columns[1]
    sample[stgt] = sample[sid].map(dict(rows)).fillna("")
    empty = (sample[stgt].str.strip() == "").sum()
    sample.to_csv(f"{WORK_DIR}/submission.csv", index=False)
    sample.to_csv("submission.csv", index=False)
    print(f"✅ Wrote submission.csv: {len(sample)} rows, {empty} empty")
else:
    # Create submission from scratch
    sub = pd.DataFrame(rows, columns=[id_col, "transcription"])
    sub.to_csv("submission.csv", index=False)
    print(f"✅ Created submission.csv with {len(sub)} rows")

print("\n" + "="*50)
print("🎉 Training and inference complete!")
print(f"Model saved at: {OUT_DIR}/final")
print("Submission saved as: submission.csv")
print("="*50)